# Laboratorio 03 — CTEs y Window Functions SQL sobre tu propio dataset

**Semana:** 03 | **Actividad de referencia:** Actividad 03  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica CTEs (`WITH`), subconsultas anidadas y funciones de ventana (`ROW_NUMBER`, `RANK`, `LAG`, `LEAD`, `SUM OVER`) de la Actividad 03 sobre tu dataset personal en Delta.

## Parte 1 — Descripción del dataset

1. **Nombre, fuente y URL** del dataset.
2. **Columna temporal:** ¿Tiene alguna columna de fecha/timestamp? ¿De qué tipo?
3. **Columna de partición:** ¿Qué columna categórica usarás como `PARTITION BY` en las window functions? ¿Por qué?
4. **Columna de orden:** ¿Qué columna usarás para `ORDER BY` dentro de la ventana?
5. **Preguntas de negocio** que respondan con ranking o análisis de tendencia.

**Escribe tu respuesta aquí:**

## Parte 2 — Cargar el dataset como tabla Delta

In [ ]:
VOL          = "/Volumes/workspace/default/week_3"
ARCHIVO      = "tu_archivo.csv"
TABLA        = "workspace.default.lab03_03_mi_dataset"

df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(f"{VOL}/{ARCHIVO}")

df.write.format("delta").mode("overwrite").saveAsTable(TABLA)
print(f"✓ Tabla: {TABLA} — {df.count():,} filas x {len(df.columns)} columnas")
df.printSchema()

## Parte 3 — Perfil técnico del dataset

In [ ]:
# Estadísticas descriptivas básicas
spark.sql(f"SELECT * FROM {TABLA} LIMIT 5").show(truncate=False)
spark.sql(f"SELECT COUNT(*) AS total, COUNT(DISTINCT columna_particion) AS grupos FROM {TABLA}").show()

In [ ]:
# Distribución de registros por columna de partición
spark.sql(f"""
    SELECT columna_particion, COUNT(*) AS registros
    FROM {TABLA}
    GROUP BY columna_particion
    ORDER BY registros DESC
    LIMIT 20
""").show(truncate=False)

**Observación:** ¿La partición está balanceada? ¿Hay grupos con muy pocos registros donde el ranking pierde sentido?

## Parte 4 — CTEs (WITH)

Aplica al menos 2 consultas que usen CTEs para organizar lógica reutilizable.

In [ ]:
# CTE 1: Preprocesamiento básico + consulta sobre el resultado del CTE
spark.sql(f"""
    WITH base_limpia AS (
        SELECT *
        FROM {TABLA}
        WHERE columna_clave IS NOT NULL
          AND columna_numerica > 0
    ),
    resumen_por_grupo AS (
        SELECT
            columna_particion,
            COUNT(*)           AS total,
            AVG(columna_numerica) AS promedio,
            MAX(columna_numerica) AS maximo
        FROM base_limpia
        GROUP BY columna_particion
    )
    SELECT *
    FROM resumen_por_grupo
    WHERE total > 5
    ORDER BY promedio DESC
    LIMIT 15
""").show(truncate=False)

**Por qué CTEs aquí:** ¿Qué ventaja tiene usar CTEs en lugar de subconsultas anidadas? ¿Cómo mejoraría la legibilidad si añadieras un tercer CTE?

In [ ]:
# CTE 2: Consulta analítica libre — diseña un pipeline SQL de 2+ pasos con CTEs
spark.sql(f"""
    WITH paso_1 AS (
        -- Escribe tu primer paso de transformación
        SELECT *
        FROM {TABLA}
    ),
    paso_2 AS (
        -- Aplica lógica sobre paso_1
        SELECT *
        FROM paso_1
    )
    SELECT * FROM paso_2 LIMIT 20
""").show(truncate=False)

**Conclusión CTE 2:**

## Parte 5 — Window Functions

In [ ]:
# ROW_NUMBER: asignar un número secuencial único dentro de cada partición
spark.sql(f"""
    SELECT
        columna_particion,
        columna_orden,
        columna_numerica,
        ROW_NUMBER() OVER (
            PARTITION BY columna_particion
            ORDER BY columna_orden DESC
        ) AS row_num
    FROM {TABLA}
    ORDER BY columna_particion, row_num
    LIMIT 30
""").show(truncate=False)

**Análisis:** Filtra con `WHERE row_num = 1` en una subconsulta o CTE. ¿Qué obtiene ese filtro en tu dataset?

In [ ]:
# Top 1 por grupo usando ROW_NUMBER en CTE
spark.sql(f"""
    WITH ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY columna_particion
                ORDER BY columna_numerica DESC
            ) AS row_num
        FROM {TABLA}
    )
    SELECT *
    FROM ranked
    WHERE row_num = 1
    ORDER BY columna_particion
""").show(truncate=False)

In [ ]:
# RANK vs DENSE_RANK: ¿qué diferencia hay cuando hay empates?
spark.sql(f"""
    SELECT
        columna_particion,
        columna_numerica,
        RANK()       OVER (PARTITION BY columna_particion ORDER BY columna_numerica DESC) AS rank_con_salto,
        DENSE_RANK() OVER (PARTITION BY columna_particion ORDER BY columna_numerica DESC) AS rank_denso
    FROM {TABLA}
    LIMIT 30
""").show(truncate=False)

**Diferencia RANK vs DENSE_RANK:** En tu dataset, ¿hay empates? ¿Cuándo usarías cada uno?

In [ ]:
# LAG y LEAD: acceder al valor anterior/siguiente dentro de la partición
spark.sql(f"""
    SELECT
        columna_particion,
        columna_orden,
        columna_numerica,
        LAG(columna_numerica,  1, 0) OVER (
            PARTITION BY columna_particion ORDER BY columna_orden
        ) AS valor_anterior,
        LEAD(columna_numerica, 1, 0) OVER (
            PARTITION BY columna_particion ORDER BY columna_orden
        ) AS valor_siguiente,
        ROUND(
            (columna_numerica - LAG(columna_numerica, 1, 0) OVER (
                PARTITION BY columna_particion ORDER BY columna_orden
            )) * 100.0 / NULLIF(LAG(columna_numerica, 1, 0) OVER (
                PARTITION BY columna_particion ORDER BY columna_orden
            ), 0), 2
        ) AS variacion_pct
    FROM {TABLA}
    ORDER BY columna_particion, columna_orden
    LIMIT 30
""").show(truncate=False)

**Análisis LAG/LEAD:** ¿Qué filas tienen `variacion_pct` más alta o baja? ¿Qué evento de negocio podría explicar ese cambio?

In [ ]:
# SUM OVER: acumulado corrido por partición
spark.sql(f"""
    SELECT
        columna_particion,
        columna_orden,
        columna_numerica,
        SUM(columna_numerica) OVER (
            PARTITION BY columna_particion
            ORDER BY columna_orden
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS acumulado
    FROM {TABLA}
    ORDER BY columna_particion, columna_orden
    LIMIT 30
""").show(truncate=False)

**Análisis acumulado:** ¿A qué porcentaje del total llega cada registro? ¿Hay algún punto de inflexión donde el acumulado crece más rápido?

## Parte 6 — Preguntas de negocio

Responde las preguntas de la Parte 1 usando CTEs y window functions.

In [ ]:
# Pregunta 1:
spark.sql(f"""

""").show(truncate=False)

**Conclusión pregunta 1:**

In [ ]:
# Pregunta 2:
spark.sql(f"""

""").show(truncate=False)

**Conclusión pregunta 2:**

In [ ]:
# Pregunta 3 (la más compleja — combina CTEs y al menos 2 window functions):
spark.sql(f"""

""").show(truncate=False)

**Conclusión pregunta 3:**

## Parte 7 — Reflexión final

1. ¿Cuándo es imprescindible usar una window function en lugar de un GROUP BY?
2. ¿Cuál es la diferencia entre `ROWS BETWEEN` y `RANGE BETWEEN` en una ventana deslizante?
3. ¿Cómo expresarías `LAG(...) OVER (PARTITION BY ... ORDER BY ...)` en PySpark? ¿La sintaxis es más o menos intuitiva?
4. ¿Qué patrón de consulta de este laboratorio reusarías en un pipeline de producción?

---

## Entrega en Git

```bash
git add semana_03/laboratorios/lab_03_sql_avanzado.ipynb
git commit -m "lab: semana03 lab03 CTEs window functions <nombre-dataset> - <tu-nombre>"
git push origin feature/semana03-sql-<tu-nombre>
```